# DBSCAN Clustering Notebook
Extracted from HTML.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

In [ ]:
X, _ = make_moons(n_samples=300, noise=0.06, random_state=42)
X_scaled = StandardScaler().fit_transform(X)

In [ ]:
minPts = 5
neighbors = NearestNeighbors(n_neighbors=minPts).fit(X_scaled)
distances, _ = neighbors.kneighbors(X_scaled)
k_distances = np.sort(distances[:, minPts - 1])[::-1]

plt.plot(k_distances)
plt.xlabel('Points sorted by distance')
plt.ylabel(f'{minPts}-th nearest neighbor distance')
plt.title('k-Distance Graph')
plt.show()

In [ ]:
dbscan = DBSCAN(eps=0.25, min_samples=minPts, metric='euclidean')
labels = dbscan.fit_predict(X_scaled)

In [ ]:
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)
print(f"Clusters found: {n_clusters}, Noise points: {n_noise}")

# Silhouette score should exclude noise points
mask = labels != -1
if n_clusters > 1:
    print("Silhouette:", silhouette_score(X_scaled[mask], labels[mask]))

In [ ]:
unique_labels = set(labels)
colors = plt.cm.get_cmap('tab10', len(unique_labels))

for k in unique_labels:
    mask = labels == k
    color = 'black' if k == -1 else colors(k)
    marker = 'x' if k == -1 else 'o'
    plt.scatter(X[mask, 0], X[mask, 1], c=[color], marker=marker, s=40, label=f'Cluster {k}' if k != -1 else 'Noise')

plt.title('DBSCAN on make_moons')
plt.legend()
plt.show()

In [ ]:
df = pd.read_csv("Mall_Customers.csv")
X_mall = df[['Annual Income (k$)', 'Spending Score (1-100)']].values
X_mall_scaled = StandardScaler().fit_transform(X_mall)

dbscan_mall = DBSCAN(eps=0.35, min_samples=5)
df['Cluster'] = dbscan_mall.fit_predict(X_mall_scaled)
df['Cluster'].value_counts()

In [ ]:
import joblib
joblib.dump(dbscan_mall, 'dbscan_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Train a classifier on DBSCAN's own output (excluding noise)
core_mask = dbscan_mall.labels_ != -1
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X_mall_scaled[core_mask], dbscan_mall.labels_[core_mask])

new_point_scaled = scaler.transform([[75, 82]])
predicted_cluster = knn.predict(new_point_scaled)